In [4]:
import torch
import torchvision

# --- MODEL NAVIGÁTOR (Klasifikace) ---
# Tento model rozhoduje, co robot vidí (Corridor, Left, Right, Křižovatky...)
model_navigator = torchvision.models.resnet18(pretrained=False)

# ZMĚNA: Musíme nastavit 10 výstupů, protože máme 10 tříd (0-9)
model_navigator.fc = torch.nn.Linear(512, 10)

In [5]:
# Načtení vah pro 10-třídní model
try:
    model_navigator.load_state_dict(torch.load('best_model_navigator_10class.pth'))
    print("Vahy úspěšně načteny.")
except FileNotFoundError:
    print("CHYBA: Soubor 'best_model_navigator_10class.pth' nenalezen! Nahrajte ho prosím.")
except RuntimeError as e:
    print(f"CHYBA v rozměrech modelu: {e}")

Vahy úspěšně načteny.


In [6]:
device = torch.device('cuda')

# Přesun modelu na grafickou kartu a přepnutí do režimu 'eval' (neuč se, jen hádej)
model_navigator = model_navigator.to(device)
model_navigator = model_navigator.eval().half() # half() zrychlí výpočet (FP16)

In [7]:
import cv2
import numpy as np

# Definujeme střední hodnotu a odchylku rovnou jako Tensory na GPU v poloviční přesnosti (FP16)
# To výrazně zrychlí výpočet, protože nemusíme data kopírovat tam a zpět
mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess(camera_value):
    global device, mean, std
    x = camera_value
    
    # 1. Konverze z BGR (OpenCV) do RGB
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
    
    # 2. Přehození os z HWC (Height, Width, Channel) na CHW (Channel, Height, Width)
    x = x.transpose((2, 0, 1))
    
    # 3. Převod z Numpy pole na Torch Tensor a přesun na GPU
    x = torch.from_numpy(x).to(device)
    
    # 4. Klíčový krok: Převod na FP16 (.half) a škálování z 0-255 na 0-1
    x = x.half().div(255.0)
    
    # 5. Normalizace (odečtení průměru a podělení odchylkou) přímo na GPU
    x = x.sub_(mean[:, None, None]).div_(std[:, None, None])
    
    # 6. Přidání batch dimenze (tvar [1, 3, 224, 224])
    x = x[None, ...]
    
    return x

In [8]:
import os
import time
import numpy as np
from PIL import Image, ImageFont, ImageDraw

# --- DŮLEŽITÉ: Názvy tříd musí přesně sedět s tréninkem (0-9) ---
# Pořadí nyní odpovídá číslování složek v 'dataset_navigator_10class'
CLASSES = [
    'corner_left',      # Index 0
    'corner_right',     # Index 1
    'corridor',         # Index 2
    'dead_end',         # Index 3
    'drift_left',       # Index 4
    'drift_right',      # Index 5
    'goal',             # Index 6
    't_junction_lr',    # Index 7
    't_junction_sl',    # Index 8
    't_junction_sr'     # Index 9
]

# Složka pro ukládání snímků (pro zpětnou analýzu)
path_to_img_folder = 'images_log'
if not os.path.exists(path_to_img_folder):
    os.makedirs(path_to_img_folder)

# Font pro text v obraze
try:
    font = ImageFont.truetype("/usr/share/fonts/dejavu/DejaVuSans.ttf", 20)
except:
    font = ImageFont.load_default()

def save_telemetry(raw_image, nav_probs, action_str):
    """
    Ukládá snímky pouze s klasifikačními daty.
    """
    img_file_nm = os.path.join(path_to_img_folder, f'img_{int(time.time() * 100)}.jpg')
    img = Image.fromarray(raw_image)
    draw = ImageDraw.Draw(img)
    width, height = img.size
    
    # 1. ZJISTĚNÍ NEJPRAVDĚPODOBNĚJŠÍ TŘÍDY
    max_idx = np.argmax(nav_probs)
    class_name = CLASSES[max_idx]
    confidence = nav_probs[max_idx]
    
    # 2. VIZUALIZACE (Text)
    # Barva textu podle jistoty (zelená = jistota, červená = nejistota)
    text_color = (0, 255, 0) if confidence > 0.7 else (255, 100, 100)
    
    draw.text((10, 10), f"VIDÍM: {class_name}", fill=text_color, font=font)
    draw.text((10, 35), f"JISTOTA: {confidence:.2f}", fill=text_color, font=font)
    draw.text((10, 60), f"AKCE: {action_str}", fill=(255, 255, 0), font=font)

    # Varování při Dead End (volitelně i pro Drift, kdyby byla jistota malá)
    if 'dead_end' in class_name and confidence > 0.8:
        draw.rectangle((0, 0, width-1, height-1), outline="red", width=5)

    img.save(img_file_nm)

In [9]:
import traitlets
from IPython.display import display
import ipywidgets.widgets as widgets
from jetbot import Camera, bgr8_to_jpeg, Robot

# 1. Inicializace Kamery
try:
    camera = Camera.instance(width=224, height=224)
except:
    print("Kamera již běží.")

image = widgets.Image(format='jpeg', width=224, height=224)

# 2. Inicializace Robota
robot = Robot()

# 3. Propojení kamery s widgetem (živý náhled)
camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

# 4. Bezpečnostní tlačítko STOP
stop_button = widgets.Button(description='STOP (Emergency)', button_style='danger')

def stop_robot_click(change):
    robot.stop()
    
stop_button.on_click(stop_robot_click)

# Zobrazení
display(widgets.HBox([image, stop_button]))

In [45]:
import torch.nn.functional as F
import time
import numpy as np

# --- 1. KALIBRACE POHYBU ---
SPEED_FORWARD = 0.1       # Rychlost jízdy rovně
SPEED_TURN = 0.25         # Rychlost motorů při otáčení

# Rychlost pro korekce (Drift) - jeden motor zrychlí, aby se robot srovnal
SPEED_CORRECT_OFFSET = 0.05 

# Časy pro provedení manévru (v sekundách)
TIME_TURN_90 = 0.35       # Otočka 90°
TIME_TURN_180 = 0.72      # Otočka 180°
TIME_NUDGE = 0.2          # Popojetí PŘED otočkou
TIME_POST_TURN = 0.8      # Popojetí PO otočce

CONF_THRESHOLD = 0.70

# --- PROMĚNNÉ STAVU ---
is_busy = False           
action_description = "IDLE"
skip_frames = 0           # Počítadlo pro zahození starých snímků po otočce

# --- TŘÍDY (ABECEDNĚ SEŘAZENÉ) ---
CLASSES = [
    'corner_left',      # 0
    'corner_right',     # 1
    'corridor',         # 2
    'dead_end',         # 3
    'drift_left',       # 4
    'drift_right',      # 5
    'goal',             # 6
    't_junction_lr',    # 7
    't_junction_sl',    # 8
    't_junction_sr'     # 9
]

# ==========================================
# A. POHYBOVÉ FUNKCE
# ==========================================

def move_forward():
    """Jede rovně."""
    # Kalibrace pro rovný směr (pravý motor +0.035)
    robot.set_motors(SPEED_FORWARD, SPEED_FORWARD + 0.035) 

def correct_course_left():
    """Jemná korekce doleva (přidáme plyn pravému motoru)."""
    # Zvýšíme pravý motor
    robot.set_motors(SPEED_FORWARD, SPEED_FORWARD + 0.035 + SPEED_CORRECT_OFFSET)

def correct_course_right():
    """Jemná korekce doprava (přidáme plyn levému motoru)."""
    # Zvýšíme levý motor
    robot.set_motors(SPEED_FORWARD + SPEED_CORRECT_OFFSET, SPEED_FORWARD + 0.035)

def stop_robot():
    """Okamžitě zastaví."""
    robot.stop()

def maneuver_turn_left():
    print("Manévr: VLEVO")
    robot.stop()
    time.sleep(0.1)
    
    # 1. Popojet do křižovatky
    move_forward()
    time.sleep(TIME_NUDGE)
    
    # 2. Otočka
    robot.left(SPEED_TURN)
    time.sleep(TIME_TURN_90)
    
    # 3. Srovnání v nové chodbě
    move_forward() 
    time.sleep(TIME_POST_TURN)
    
    robot.stop()
    time.sleep(0.1)

def maneuver_turn_right():
    print("Manévr: VPRAVO")
    robot.stop()
    time.sleep(0.1)
    
    # 1. Popojet
    move_forward()
    time.sleep(TIME_NUDGE)
    
    # 2. Otočka (s kompenzací času, pokud je potřeba)
    robot.right(SPEED_TURN)
    time.sleep(TIME_TURN_90 - 0.035)
    
    # 3. Srovnání
    move_forward()
    time.sleep(TIME_POST_TURN)
    
    robot.stop()
    time.sleep(0.1)

def maneuver_turn_around():
    print("Manévr: ČELEM VZAD")
    robot.stop()
    time.sleep(0.2)
    
    # Otočka
    robot.left(SPEED_TURN)
    time.sleep(TIME_TURN_180)
    
    robot.stop()
    time.sleep(0.2)

# ==========================================
# B. LOGIKA PRAVÉ RUKY (+ DRIFT)
# ==========================================

def get_right_hand_action(class_idx):
    """
    Mapování (0-9):
    0:corn_L, 1:corn_R, 2:corridor, 3:dead, 
    4:drift_L, 5:drift_R, 6:goal, 
    7:t_LR, 8:t_SL, 9:t_SR
    """
    
    if class_idx == 0: return "TURN_LEFT"       # Corner Left
    if class_idx == 1: return "TURN_RIGHT"      # Corner Right
    if class_idx == 2: return "FORWARD"         # Corridor
    if class_idx == 3: return "TURN_AROUND"     # Dead End
    
    # --- KOREKCE ---
    # Drift Left (jsem vlevo) -> Musím doprava
    if class_idx == 4: return "CORRECT_RIGHT"
    # Drift Right (jsem vpravo) -> Musím doleva
    if class_idx == 5: return "CORRECT_LEFT"
    
    if class_idx == 6: return "GOAL"            # Goal
    
    # --- KŘIŽOVATKY ---
    if class_idx == 7: return "TURN_RIGHT"      # T-Junction LR
    if class_idx == 8: return "FORWARD"         # T-Junction SL -> ROVNĚ!
    if class_idx == 9: return "TURN_RIGHT"      # T-Junction SR
    
    return "STOP"

# ==========================================
# C. HLAVNÍ SMYČKA
# ==========================================

def execute(change):
    global is_busy, action_description, skip_frames
    
    # 1. ČIŠTĚNÍ BUFFERU (Zahození starých snímků po manévru)
    if skip_frames > 0:
        skip_frames -= 1
        return

    # Pokud robot zrovna točí, nekoukáme na kameru
    if is_busy:
        return

    # 2. Zpracování obrazu
    image = change['new']
    image_input = preprocess(image)

    # 3. Inference
    nav_output = model_navigator(image_input)
    nav_probs = F.softmax(nav_output, dim=1).flatten()
    
    probs = nav_probs.detach().cpu().numpy()
    class_idx = int(np.argmax(probs))
    confidence = probs[class_idx]
    
    # 4. Rozhodnutí
    if confidence < CONF_THRESHOLD:
        action = "WAIT"
        robot.stop()
    else:
        action = get_right_hand_action(class_idx)
    
    action_description = f"{CLASSES[class_idx]} ({confidence:.2f}) -> {action}"
    
    # 5. Provedení akce
    if action == "FORWARD":
        move_forward()

    elif action == "CORRECT_LEFT":
        # Jemná korekce doleva za jízdy (nezastavujeme)
        correct_course_left()
        
    elif action == "CORRECT_RIGHT":
        # Jemná korekce doprava za jízdy (nezastavujeme)
        correct_course_right()
        
    elif action == "GOAL":
        robot.stop()
        print("CÍL NALEZEN! Mise dokončena.")
        is_busy = True 
        return

    elif action in ["TURN_LEFT", "TURN_RIGHT", "TURN_AROUND"]:
        is_busy = True 
        
        # Provedení blokujícího manévru
        if action == "TURN_LEFT":
            maneuver_turn_left()
        elif action == "TURN_RIGHT":
            maneuver_turn_right()
        elif action == "TURN_AROUND":
            maneuver_turn_around()
            
        is_busy = False 
        
        # DŮLEŽITÉ: Po manévru zahodíme příštích 20 snímků!
        skip_frames = 20 
        
        # Po manévru se rozjedeme rovně
        move_forward()
        
    elif action == "WAIT":
        robot.stop()

    # Telemetrie (volitelné)
    save_telemetry(image, probs, action)

In [11]:
# Spustit smyčku
execute({'new': camera.value}) # Inicializační průchod
robot.stop()
print("START: Robot jede podle pravidla pravé ruky!")

START: Robot jede podle pravidla pravé ruky!


In [58]:
# Připojíme funkci 'execute' ke kameře
# Tím se spustí smyčka řízení v reálném čase
camera.observe(execute, names='value')

Manévr: ČELEM VZAD
Manévr: VPRAVO
Manévr: VPRAVO
Manévr: VLEVO
Manévr: ČELEM VZAD
Manévr: ČELEM VZAD
Manévr: ČELEM VZAD


In [59]:
camera.unobserve(execute, names='value')
robot.stop()
is_busy = False
print("STOP: Robot zastaven.")


STOP: Robot zastaven.


In [50]:
import glob
import cv2
import os

def create_ordered_img_array():
    img_name_array = []
    # ZMĚNA: Hledáme koncovku .jpg (shoda s funkcí save_frames)
    for filename in glob.glob('images_log/*.jpg'):
        img_name_array.append(filename)
    
    # Seřadíme podle názvu (jelikož názvy obsahují čas, seřadí se chronologicky)
    img_name_array.sort()

    img_array = []
    for filename in img_name_array:    
        img = cv2.imread(filename)
        if img is not None: # Kontrola, zda se obrázek načetl
            height, width, layers = img.shape
            size = (width,height)
            img_array.append(img)
            
    print(f'{len(img_array)} snímků nahráno pro video.')    
    return img_array   

def make_video(video_file_name, array_of_images, fps=15, image_size=(224,224)):
    if len(array_of_images) == 0:
        print("Varování: Žádné snímky pro vytvoření videa.")
        return

    # Používáme kodek DIVX (standard pro .avi)
    out = cv2.VideoWriter(video_file_name, cv2.VideoWriter_fourcc(*'DIVX'), fps, image_size)
    for i in range(len(array_of_images)):
        out.write(array_of_images[i])
    out.release()
    print(f"Video {video_file_name} uloženo.")
    
def delete_images():
    # ZMĚNA: Mažeme soubory .jpg
    count = 0
    if not os.path.exists('images_log'):
        return
        
    for image_file_name in os.listdir('images_log'):
        if image_file_name.endswith(".jpg"):
            os.remove('images_log/' + image_file_name)
            count += 1
    print(f"Smazáno {count} dočasných snímků.")

In [60]:
# Create array of images
img_array = create_ordered_img_array()

# Make video with 1 fps
make_video('video_1_fps.avi',img_array,1)

# Make video with 15 fps (actual speed)
make_video('video_15_fps.avi',img_array,15)

1739 snímků nahráno pro video.
Video video_1_fps.avi uloženo.
Video video_15_fps.avi uloženo.


In [61]:
# Delete all images (Clean image folder)
delete_images()


Smazáno 1739 dočasných snímků.
